# Some Standardized shocks to illustrate the main macroeconomic features of the model

This notebook performs a set of standard macroeconomic shocks on a model and displays results.  It can be used to verify the good functioning of the model and also illustrates nicely how to perform simulations on World Bank models in python using the `modelflow` package.

The simulations performed are:

* 5 separate stimulus shocks equal to 1 percent of *ex ante* GDP.  These can be used to compare fiscal multipliers and the impacts of different kinds of stimulus on GDP, potential GDP, consumption and inflation
    * a 1 percent of GDP decrease in indirect taxes
    * a 1 percent of GDP decrease in direct taxes
    * a 1 percent of GDP increase in government spending on goods and services
    * a 1 percent of GDP increase in government spending on investment goods
    * a 1 percent of GDP increase in government spending on transfers to households
* 4 non-fiscal shocks were also run
    * a temporary one-year 1 percent increase in the monetary policy interest rate;
    * a permanent 10 percent depreciation of the currency
    * a permanent one percent increase in total factor productivity;
    * a permanent \$20 increase in the price of crude oil

**Note:**
This Notebook is designed to be run on any World Bank model. To customize the notebook to run on a different model, the string variable (`Cty`), which is defined in section 1.1 will have to be changed to take the value of the Mnemonic of the country to be simulated. Similarly the location of the file containing the model may have to be revised.   While  the World Bank mnemonics are the same across countries, not all countries report all variables.  As a result for some models, some variable names (notably those of shocked variables or the expenditure variables being held constant) may need to be revised. Otherwise the Notebook should run without change on any World Bank model.

## Set up python environment and load model object

To work with modelflow we must first import the python libraries that we wish to work with and then instantiate the model object, which we have chose to call `themodel`;

In [1]:
from modelclass import model

In [2]:
#Jupyter notebook code that improves the look of the executed notebook
%load_ext autoreload
%autoreload 2

In [3]:
#This is code to manage dependencies if the notebook is executed in the Google Colab cloud service
if 'google.colab' in str(get_ipython()):
  import os
  os.system('apt -qqq install graphviz')
  os.system('pip -qqq install ModelFlowIb   ')


In [4]:
#Set this variable to the three-letter ISO of the country whose model is being simulated
cty="irq" 
# Models downloaded from the World Bank web site using the model.download_github_repo(reset=1) method of modelflow and
# executing this file from the local version of the file stored on their computer shuold not have to changethe filepath varible
filepath=f'data/{cty}'
themodel,bline = model.modelload(filepath,run=True,keep='Baseline') 

Zipped file read:  data\irq.pcim


## Prepare the simulations

For each shock, a separate `DataFrame` is created. Each of these `DataFrames` is given a name that evokes the shock to be performed. Then each `DataFrame` is modified to reflect the shock that is to be performed.

Following the creation of the `DataFrames` the shocks will be performed and the results stored using the `keep=` syntax of model flow.

### Fiscal policy shocks

If necessary, the two lines below can be uncommented in order to generate a list of all variables in the model that start GGEXP (general government expenditure) and GGREV (general government revenues) and that end CN (millions of current local currency units).

In [5]:
print("Expenditure variables")
themodel['???GGEXP*CN'].des #Uncomment to get list of mnemonics and descriptions for government expenditures
print("")
print("Revenue variables")
themodel['???GGREV*CN'].des #Uncomment to get list of mnemonics and descriptions for government revenues

Expenditure variables
IRQGGEXPCAPECN : Capital_expenditure
IRQGGEXPCAPTCN : Government_gross_fixed_investment
IRQGGEXPCRNTCN : Current_expenditure
IRQGGEXPGNFSCN : Expenditures_on_goods_and_services
IRQGGEXPINTDCN : Interest_payments_on_domestic_debt
IRQGGEXPINTECN : Interest_payments_on_external_debt
IRQGGEXPINTPCN : Interest_payments
IRQGGEXPMAINCN : Maintenance_of_assets
IRQGGEXPOTHRCN : Other_expenditures
IRQGGEXPSPECCN : Special_programs
IRQGGEXPSUBTCN : Subsidies
IRQGGEXPTOTLCN : Total_expenditures
IRQGGEXPTRNSCN : Social_welfare_transfers
IRQGGEXPWAGECN : Wage_Compensation

Revenue variables
IRQGGREVCOMMCN : Oil_revenues
IRQGGREVDRCTCN : Direct_taxes_(income_tax)
IRQGGREVGNFSCN : Taxes_on_goods_services_(consumption_tax)
IRQGGREVNCOMCN : Non_oil_revenues
IRQGGREVNTAXCN : Non_tax_revenues
IRQGGREVTAXTCN : Tax_revenues
IRQGGREVTOTLCN : Total_revenues


#### Create an expenditures string

The fiscal scenarios below exogenize (hold constant) spending on those elements of government spending that are not being directly shocked.

To facilitate that, the variable `GGexp` is assigned a string containing all of the expenditure variables that are to be held constant.  This variable is then used when setting up each of the fiscal shocks below. 

This list may need to be adjusted from model to model.

In [6]:
# Government spending variables to be held constant
GGexp=f'{cty}GGEXPCAPTCN {cty}GGEXPGNFSCN {cty}GGEXPOTHRCN' + \
      f'{cty}GGEXPTOTHCN {cty}GGEXPTRNSCN {cty}GGEXPTOTSCN ' + \
      f'{cty}GGEXPSUBSCN {cty}GGEXPWAGECN';

### The Indirect tax cut

This shock assumes that the main elements of government spending are held constant at their pre-shock levels.  This assumption could be relaxed by commenting out the second line. 

In the model, indirect taxes are determined as a function of their lagged effective tax rate (Revenues divided by assumed tax base).  

Inspecting the Goods and services tax revenue equation, one can note that the first term drops out in the forecast period (DUMH=0 in the forecast period) and the equation resolves into an identity where revenues evolve according to the tax base and the effective tax rate of the previous year.

To shock the level of indirect spending revenues in 2025 the add factor is used.  By reducing revenues in 2025 by 1 percent of GDP, the effective tax rate in that year falls.  For subsequent year's it is this lower effective rate that is retained, thereby making the shock permanent.

In [7]:
fpol_indirect=bline.copy()
fpol_indirect=themodel.fix(bline,f'{GGexp}') # Freeze other spending levels

# PIT and CIT tax revenues are determined by the lagged effective rate.  
# Thus shocking the level of tax revenues in 2025 by one percent of GDP
# will increase the effective tax rate going forward by a constant amount
# consistent with a .5 percent increase in GDP and the tax base for consumption
fpol_indirect=fpol_indirect.mfcalc(
    f'<2025 2025> {cty}GGREVGNFSCN_A ={cty}GGREVGNFSCN_A - .01*{cty}NYGDPMKTPCN')


#solve the model.
tempdf = themodel(fpol_indirect,silent=1,keep=f'1 % of GDP Indirect tax cut')

The folowing variables are fixed
IRQGGEXPCAPTCN
IRQGGEXPGNFSCN
IRQGGEXPTRNSCN
IRQGGEXPWAGECN


### Direct tax hike of 1% of GDP

The same basic methodology is followed for  direct taxes.


In [8]:
fpol_direct=bline.copy()
fpol_direct=themodel.fix(bline,f'{GGexp}') # Freeze spending levels Capital

# In the model direct taxes (personal and corporate income taxes) tax are 
# determined by the lagged effective rate of taxation.  Shocking the 
# level of tax revenues in 2025 by one percent of GDP decreases the effective 
# tax rate going forward by a constant amount consistent with a 1 percent decrease 
# in GDP and the tax base for direct taxes (.5% for each)
fpol_direct=fpol_direct.mfcalc(
    f'<2025 2025> {cty}GGREVDRCTCN_A ={cty}GGREVDRCTCN_A  - .01*{cty}NYGDPMKTPCN')

#solve the model.
tempdf = themodel(fpol_direct,silent=1,keep=f'1 % of GDP direct tax cut')

The folowing variables are fixed
IRQGGEXPCAPTCN
IRQGGEXPGNFSCN
IRQGGEXPTRNSCN
IRQGGEXPWAGECN


### Increase in expenditure on goods and services

The ex ante fiscal effort is the same in this scenario (1% of ex ante GDP) with the difference that it is implemented as an increase government spending, in this instance on goods and services.



In [9]:
fpol_ExpGS=bline.copy()
fpol_ExpGS=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

fpol_ExpGS=fpol_ExpGS.mfcalc(
    f'<2025 2050> {cty}GGEXPGNFSCN_X ={cty}GGEXPGNFSCN_X   + .01*{cty}NYGDPMKTPCN')


The folowing variables are fixed
IRQGGEXPCAPTCN
IRQGGEXPGNFSCN
IRQGGEXPTRNSCN
IRQGGEXPWAGECN


In [10]:
#solve the model.
tempdf = themodel(fpol_ExpGS,silent=1,keep=f'1 % of GDP increase in G&S spending')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']

### Increase in expenditure on investment goods

The ex ante fiscal effort is the same in this scenario (1% of ex ante GDP), implemented as an increase in government spending on capital goods. 


In [11]:
fpol_ExpInv=bline.copy()
fpol_ExpInv=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

fpol_ExpInv=fpol_ExpInv.mfcalc(f'<2025 2050> {cty}GGEXPCAPTCN_X ={cty}GGEXPCAPTCN_X   + .01*{cty}NYGDPMKTPCN')

The folowing variables are fixed
IRQGGEXPCAPTCN
IRQGGEXPGNFSCN
IRQGGEXPTRNSCN
IRQGGEXPWAGECN


In [12]:
#solve the model.
tempdf = themodel(fpol_ExpInv,silent=1,keep=f'1 % of GDP increase in Govt investment spending')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



### Increase in expenditure on Transfers to households

In this scenario, the same fiscal effort is implemented as an increase in transfers to households.


In [13]:
fpol_ExpTrans=bline.copy()
fpol_ExpTrans=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

fpol_ExpTrans=fpol_ExpTrans.mfcalc(f'<2025 2050> {cty}GGEXPTSOCCN_X ={cty}GGEXPTSOCCN_X   + .01*{cty}NYGDPMKTPCN')

The folowing variables are fixed
IRQGGEXPCAPTCN
IRQGGEXPGNFSCN
IRQGGEXPTRNSCN
IRQGGEXPWAGECN


In [14]:
#solve the model.
tempdf = themodel(fpol_ExpTrans,silent=1,keep=f'1 % of GDP increase inransfers to households')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']

## Comparisons of results from the fiscal scenarios

The following charts compare results from the different fiscal simulations. Impacts will differ both in terms of their long-term and short-term impacts.  For example a scenario that increased investment would likely have negative impacts on consumption in the short-run but in the longer run could be expected to have an opposite impact on potential output, GDP and perhaps consumption.  A scenario that concentrated on transfers or consumption might have more of a short-term impact on demand but in the long run would have limited (and potentially negative impacts on output), especially if increased fiscal deficits and debt crowded out private sector investment.  As all World Bank models are customized to the country for which they have been built the extent of these effects can vary across models.


Recall the scenarios that were run by interrogating the keep_solutions dictionary.



### Real GDP impacts and impacts on main Real GDP expenditure components

The following ploy uses the `scenarios=` option clause to select which scenario results are to be plotted, and then plots them on a series of charts one one page using the `samefig=True` and the `datatype=difpctlevel` options.  The first option ensures that all the graphs are arranged in a grid and the second expresses the results as a percent deviation from the results in the first scenario specified in the `scenarios` option -- in this case the baseline scenario. 



In [15]:
keeps=""
for key,value in themodel.keep_solutions.items():
    print("'"+key+"|'") #print out can be cut and paste into option scenarios=
    keeps=keeps+key+'|' # or keeps variable can be used
    
    
    

'Baseline|'
'1 % of GDP Indirect tax cut|'
'1 % of GDP direct tax cut|'
'1 % of GDP increase in G&S spending|'
'1 % of GDP increase in Govt investment spending|'
'1 % of GDP increase inransfers to households|'


Note, the above loop creates a string variable `keeps` which is comprised of all the scenarios -- identified by the text used in the initial keep command -- and separated by a horizontal line "|".  The string could also be generated manually (and edited as necessary) by using the output of the print statement.

The `scenario=keeps` command instructs the plot routine to show the results from the specified scenarios.  In this instance because the `datatype` is a `difpctlevel`, all results are expressed as a percent deviation in the level of the variable versus its level in the first scenario in the `keeps` string (Baseline in this instance).

In [16]:
dispvars=f'{cty}NYGDPPOTLKN {cty}NYGDPMKTPKN {cty}NECONPRVTKN {cty}NEGDIFTOTKN {cty}NEEXPGNFSKN {cty}NEIMPGNFSKN'
themodel.plot(dispvars,
              scenarios=keeps,
              start=2020, end=2040,
              datatype='difpctlevel',samefig=True,legend=True,
              title="Comparison of fiscal scenarios",name="FiscalScenarios")


Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…


For Iraq, the GDP results appear to be broadly consistent with expectations.  Fiscal expansion of all types boosted demand and GDP in the short run (second and third charts).  However, the long run the impact depends on the impact of the spending on potential output (first chart).  

Sustained increased investment spending served to increase the capital stock and contribute to higher potential and actual GDP.  Spending that focused on consumption or direct tax cuts has little discernible (or negative) impact on potential. 

Decreased direct taxation tends to benefit households by increasing disposable income, but the benefits in terms of GDP declines over time due to the relative stability of potential output and a sharp decline in next exports as the additional domestic demand is met by imports.  

The cut in indirect taxes tends to generate small but persistent benefits, presumably due to lower prices for domestically produced goods which benefits both exports and import competing goods.

### Impacts on the fiscal accounts

The impacts on several of the main fiscal indicators are illustrated below, where in this chart it is **nominal** GDP that is presented not real GDP as above.   These results are being presented as the change in the level as a percent of the pre-shock level.  In so far as the shocks are inflationary nominal GDP and both revenues and expenditures will be higher even if as a percent of GDP they may have declined.

In [17]:
themodel.plot(dispvars,
              scenarios=keeps,
              start=2020, end=2040,datatype='difpctlevel',
              samefig=True,legend=True,title="Comparison of fiscal scenarios",
              name="FiscalScenarios")

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

Note that:
* Nominal GDP is increased in all scenarios, mainly reflecting the inflationary impact of the scenario (most scenarios saw real GDP decline).  
* The fiscal account deteriorates in all scenarios as compared with the baseline.
    * Nominal spending increases even in the tax scenarios, but here the driver is increased interest payments as other elements of spending were held constant.
    * Although interest payments as a percent of their initial level are up a lot, the increase as a percent of GDP (see next set of charts is less pronounced).
    * Revenues improve in the spending scenarios because of higher nominal GDP.
    * Because the fiscal balance is negative in the baseline a deterioration appears as a positive percent change in these graphs. The percent increase in the deficit (decline in the balance) is smallest in the scenarios where real GDP growth is increasing.
    * Nominal public debt is higher in all scenarios

#### Fiscal impacts as a percent of GDP

As noted, higher inflation (due to increased demand in the early years of the simulation) mean that both revenues and expenditures are higher in the simulation scenarios.  

The following charts show the same results expressed as a percent of GDP.  Here the inflation influences both the numerator and the denominator, so just the net effect is drawn.


In [18]:
# Uncomment to display the mnemonics and desriptions of the general gobernment (GG) expenditure items 
# (EXP) in nominal local currency (CN)
#themodel[f'{cty}GGEXP*CN'].des  

In [19]:
rev_vars=f'{cty}GGREVTOTLCN {cty}GGEXPTOTLCN {cty}GGEXPINTPCN {cty}GGBALOVRLCN {cty}GGDBTTOTLCN'
themodel.plot(rev_vars,
              scenarios=keeps,
              start=2023, end=2040,
              datatype='difgdppct',
              samefig=True,legend=True,
              title="Comparison of fiscal scenarios",
              name="FiscalScenarios")

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

* Government revenues as a percent of GDP are lower than in the baseline in all scenarios, but less so in the expenditure scenarios.  The percent decline in the revenue scenarios reflects the importance of oil revenues whose nominal value does not react to the higher inflation given oil is priced in dollars and the fixed exchange rate.
* Spending as a percent of GDP increases by as much as 1.2 percent of GDP in the goods and services spending scenario, the result of the original bump in spending plus increased interest payments as the debt rises.
* In the investment scenario, the increased potential output reduces the overall impact of the spending as a share of nominal GDP.

* Interest payments rise by close to 0.2 percent of GDP by the end of the period in the indirect tax cuts and spending scenario due to higher debt levels and because of higher interest rates as debt to GDP ratios rise.
* Higher debt and fiscal borrowing will translate into increased competition for domestic and foreign savings, crowding out private sector investment
* The fiscal balance deteriorates by between 1.8 and 0.5  percent of GDP, with differences reflecting differences in real GDP and inflation impacts across scenarios.
* Debt rises by as much as 12 percent of GDP as the permanent 1 percent increase in spending accumulates over time.

## Non fiscal simulations

Three non-fiscal scenarios were run.  The first a temporary increase in the monetary policy interest rate, the second a 10 percent depreciation and the final a permanent $20 increase in the price of crude_petrol.

### Monetary policy shock

In this shock, it is assumed that the central bank raises its policy rate by 1 percentage point for 1 year.


In [20]:
Mpol=bline.copy()
Mpol=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

Mpol=themodel.fix(Mpol,f'{cty}FMLBLPOLYFR',2025,2025) # One year shock to MP but then let the equation take over.
Mpol=Mpol.mfcalc(f'<2025 2025> {cty}FMLBLPOLYFR_X ={cty}FMLBLPOLYFR_X   + 1')
#Mpol[f'{cty}FMLBLPOLYFR_X'].loc[2020:2030]/Mpol[f'{cty}FMLBLPOLYFR'].loc[2020:2030]

The folowing variables are fixed
IRQGGEXPCAPTCN
IRQGGEXPGNFSCN
IRQGGEXPTRNSCN
IRQGGEXPWAGECN
The folowing variables are fixed
IRQFMLBLPOLYFR


In [21]:
#solve the model.
tempdf = themodel(Mpol,silent=1,keep=f'1 ppt increase in policy rate in 2025')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']

themodel[f'{cty}FMLBLPOLYFR'].frml

IRQFMLBLPOLYFR : FRML <DAMP,STOC> IRQFMLBLPOLYFR = (100*IRQFMLBLPOLYFR_A+100* (0.85*IRQFMLBLPOLYFR(-1)/100+(1-0.85)*(0.0249727052023416+1.5*(((LOG(IRQNECONPRVTXN))-(LOG(IRQNECONPRVTXN(-1))))-IRQINFLEXPT/100)+0.3*IRQNYGDPGAP_/100)+0.0658396176049064*DURING_2006-0.0687072291239696*DURING_2009) ) * (1-IRQFMLBLPOLYFR_D)+ IRQFMLBLPOLYFR_X*IRQFMLBLPOLYFR_D $


In [22]:
#themodel[f'{cty}FMLBLPOLYFR']
with themodel.set_smpl(2020,2030):
    tab_t = themodel.table(name='Change in policy rate',
                           pat='{cty}FMLBLPOLYFR',title='Policy rate',datatype='diflevel',
                           foot='Source: World Bank ',transpose = True)
tab_t.show

Policy rate
     Policy_Interest_Rate_(Percent)
         --- Impact, Level ---     
2020                          -0.00
2021                          -0.00
2022                          -0.00
2023                          -0.00
2024                          -0.00
2025                           1.00
2026                           0.67
2027                           0.48
2028                           0.35
2029                           0.26
2030                           0.20
Source: World Bank 


### Exchange rate depreciation

This shock assumes a permanent depreciation of the currency by 10 percent in 2025.


In [23]:
Mpol_exr=bline.copy()
#Mpol_exr=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

Mpol_exr=themodel.fix(Mpol,f'{cty}PANUSATLS',2025,2050) # One year shock to MP but then let the equation take over.
Mpol_exr=Mpol_exr.mfcalc(f'<2025 2050> {cty}PANUSATLS_X ={cty}PANUSATLS_X * 1.1')


The folowing variables are fixed
IRQPANUSATLS


In [24]:
#solve the model.
tempdf = themodel(Mpol_exr,silent=1,keep=f'A permanent 10 percent depreciation in 2025')


### TFP Shock

This shock explores the effect of a permanent increase in the level of non-oil TFP by 1 percent beginning in 2025.


In [25]:
TFP=bline.copy()
TFP=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

#TFP=themodel.fix(Mpol,f'{cty}PANUSATLS',2025,2050) # One year shock to MP but then let the equation take over.
TFP=TFP.mfcalc(f'<2025 2050> {cty}NYGDPTFP ={cty}NYGDPTFP * 1.01')


The folowing variables are fixed
IRQGGEXPCAPTCN
IRQGGEXPGNFSCN
IRQGGEXPTRNSCN
IRQGGEXPWAGECN


In [26]:
#solve the model.
tempdf = themodel(TFP,silent=1,keep=f'A permanent 1 percent increase in TFP levels')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



### A permanent 20 percent increase in oil prices

This shock explores the sensitivity of the model to a permanent 20 USD increase in global oil and natural gas prices beginning in 2025.  The natural gas price are assumed to rise proportionately to a 20 USD increase in world crude oil prices.


In [27]:
themodel['WLDF*crude*'].des

WLDFCRUDE_PETRO            : Price of oil (USD)
WLDFCRUDE_PETRO_VALUE_2015 : WLDFCRUDE_PETRO_VALUE_2015


In [28]:
Oil=bline.copy()
Oil=themodel.fix(bline,f'{GGexp}') # Freeze spending levels 

# scenario to raise oil prices.  Assume that natural gas prices increase
# by a similar margin as a $20 increase in crude oil prices
Oil=Oil.mfcalc(
    f'<2025 2050> WLDFNHAS_EUR = WLDFNHAS_EUR * (WLDFCRUDE_PETRO+20)/WLDFCRUDE_PETRO')
Oil=Oil.mfcalc(    
    f'<2025 2050> WLDFCRUDE_PETRO = WLDFCRUDE_PETRO + 20')
Oil=Oil.mfcalc(    
    f'<2025 2050> WLDFCRUDE_PETRO_VALUE_2015 =  WLDFCRUDE_PETRO_VALUE_2015*(1+ 20/wldfcrude_petro)')



The folowing variables are fixed
IRQGGEXPCAPTCN
IRQGGEXPGNFSCN
IRQGGEXPTRNSCN
IRQGGEXPWAGECN


In [29]:
#solve the model.
tempdf = themodel(Oil,silent=1,keep=f'A permanent $20 increase in global crude oil prices')
#themodel.lastdf['IDNNECONGOVTCN']/themodel.basedf['IDNNECONGOVTCN']



### Summary  impacts of non-fiscal scenarios

The following graphs show the change in the main macroeconomic indicators (Inflation, proxied here by the inflation rate of household consumption), real GDP, and changes in the current account, fiscal balances and debt levels expressed as a percent of GDP.

In [30]:
for key,value in themodel.keep_solutions.items():
    print(f'\'{key}|\' + \\')

'Baseline|' + \
'1 % of GDP Indirect tax cut|' + \
'1 % of GDP direct tax cut|' + \
'1 % of GDP increase in G&S spending|' + \
'1 % of GDP increase in Govt investment spending|' + \
'1 % of GDP increase inransfers to households|' + \
'1 ppt increase in policy rate in 2025|' + \
'A permanent 10 percent depreciation in 2025|' + \
'A permanent 1 percent increase in TFP levels|' + \
'A permanent $20 increase in global crude oil prices|' + \


In [31]:
# In this instance we do not want to include results from the fiscal scenarios so we cut and paste the 
# desired lines generated above to set the scenarios 

scen = 'Baseline|' +\
        '1 ppt increase in policy rate in 2025|' + \
        'A permanent 10 percent depreciation in 2025|' + \
        'A permanent 1 percent increase in TFP levels|' + \
        'A permanent $20 increase in global crude oil prices|' 
        
fig1=themodel.plot(f'{cty}NECONPRVTXN',scenarios=scen,start=2023,end=2040,datatype="difgrowth",heading="Change in inflation",title="Inflation)");
fig2=themodel.plot(f'{cty}NYGDPMKTPKN {cty}NYGDPPOTLKN {cty}NEGDIFTOTKN',scenarios=scen,start=2023,end=2040,datatype="difpctlevel",heading="Percent change from baseline");
fig3=themodel.plot(f'{cty}BNCABFUNDCD {cty}GGBALOVRLCN {cty}GGDBTTOTLCN',scenarios=scen,start=2023,end=2040,datatype="difgdppct",heading="Change as a percent of GDP");

combo=(fig1|fig2|fig3).set_options(samefig=True,name='Non_fisc')
combo.show
                               

<Figure size 2000x2400 with 9 Axes>

As might be expected, effects across these scenarios are more divergent.  

**Inflation**

The exchange rate depreciation is inflationary at least in the short-run, as the costs of imported goods rise.  While the price level remains elevated, the pass through of the higher costs to inflation in this scenario is very fast.

Both the TFP shock and the monetary policy tightening contribute to declines in inflation.  The TFP shock is deflationary because it raises potential output and therefore open up a negative output gap, at least initially.  Over time this effect diminishes. The monetary policy shock also induces an increase in the output gap (due to lower demand) in the short run. In the longer run additional investment induced by high interest rates causes potential output to rise persisting the negative output gap. In both cases, as the economy adjusts inflation tends to recover its pre-shock inflation rate.

The oil price scenario has limited effects on domestic prices, in part because domestic oil prices are administered and therefore the sensitivity of domestic prices to the international price of oil is low.  

**Real side effects** 

The permanent increase in TFP translates fairly quickly into higher GDP, and GDP continues to rise -- reflecting stronger potential output.  While the initial increase in TFP will have boosted potential the subsequent increase reflects higher investment induced by the increased productivity of capital (and labor).

The exchange rate depreciation and the higher TFP boost domestic revenues (a given $ of oil exports generates more local currency revenues) and they also boost the competitiveness of domestic industry contributing to an improved current account.

Revenue and budget balance effects are generally positive, with the exception of in the interest rate shock, where higher interest rates increase government debt financing costs and contribute to a modest deterioration in the fiscal position.


#### Real GDP impacts of non-fiscal simulations

In [32]:
# Because we are showing variables as a percent of GDP itmakes no sense to show GDP as % of itslef (always 100)
dispvars_=dispvars.replace(f'{cty}NYGDPMKTPKN','') 
themodel.plot(dispvars_,
              scenarios=keeps,start=2020, end=2040,
              datatype="difgdppct",
              samefig=True,legend=True,
              title="Real GDP impacts of non-fiscal scenarios")

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

#### Discussion of real GDP impacts of non-fiscal scenarios

##### The TFP shock (green line)
Potential and real GDP impacts mirror one another, with the TFP shock raising potential output permanently, and with real GDP catching up over-time slowly.  The potential GDP impact rises proportionately over time because the higher output induces additional investment which adds further to potential GDP.  Higher potential and actual GDP translates into increased consumption, exports and imports as the economy adjusts to the higher scale of activity.

##### The monetary policy shock (blue line)
As can be expected a tightening of monetary policy has negative effects on GDP initially. Higher interest rates and slower growth reduce investment growth. Over the longer run as interest rates deckline the impact on output reverses itself.   Consumer demand  (and imports) mirror GDP as incomes declien initially but then increase as compared with the baseline.  Exports are hurt initially due to high capital costs. 

##### Depreciation (orange line)
The long run effect of the 10 percent depreciation is negative, reflecting higher import costs, which dampen investment and contribute to a cumulative 2 percent decline in potential output.  Exports benefit initially as domestic goods become more competitive abroad and imports decline as import-competing firms benefit.  However, these gains weaken over time as higher import costs translate into lower real incomes and lower consumption.

##### Oil price hike (red line)

A permanent $20 nominal increase in oil prices has modest impacts on GDP, mainly because the quantity of oil produced in teh model is exogenousand teh price effect is limited to government revenues and indirect effects through inflationm budgetary and exchange rate effects.  Overall real exports do not change because the quantity of oil exported does not change (although as noted above the value of exports do).




### Fiscal impacts of non-fiscal scenarios

The depreciation fall in spending is kind of hard to explain.

In [33]:
# Uncomment to display the mnemonics and desriptions of the general government (GG) expenditure items 
# (EXP) in nominal local currency (CN)
themodel[f'{cty}GGEXP*CN'].des  

IRQGGEXPCAPECN : Capital_expenditure
IRQGGEXPCAPTCN : Government_gross_fixed_investment
IRQGGEXPCRNTCN : Current_expenditure
IRQGGEXPGNFSCN : Expenditures_on_goods_and_services
IRQGGEXPINTDCN : Interest_payments_on_domestic_debt
IRQGGEXPINTECN : Interest_payments_on_external_debt
IRQGGEXPINTPCN : Interest_payments
IRQGGEXPMAINCN : Maintenance_of_assets
IRQGGEXPOTHRCN : Other_expenditures
IRQGGEXPSPECCN : Special_programs
IRQGGEXPSUBTCN : Subsidies
IRQGGEXPTOTLCN : Total_expenditures
IRQGGEXPTRNSCN : Social_welfare_transfers
IRQGGEXPWAGECN : Wage_Compensation


In [34]:
themodel.plot(f'{cty}GGBALOVRLCN {cty}GGEXPTOTLCN {cty}GGREVTOTLCN {cty}GGEXPINTPCN {cty}GGDBTTOTLCN',
              scenarios=scen,start=2020, end=2040,datatype="difgdppct",samefig=True,legend=True,
              title="Revenue impacts of non-fiscal scenarios",name="NonFiscRevenues")

Accordion(children=(HTML(value='<?xml version="1.0" encoding="utf-8" standalone="no"?>\n<!DOCTYPE svg PUBLIC "…

**The oil shock** has large positive impacts on the fiscal accounts, boosting revenues by more than 4 percent of GDP and the fiscal balance by almost 8 percent of GDP. Nominal fiscal expenditures are being held constant in the scenario, with the reduction as a share of GDP reflecting higher nominal GDP. 

The **depreciation scenario** also contributes to an increase in the fiscal balance as the local currency value of oil exports and the associated government revenues rise.

The **TFP shock** is broadly fiscal neutral as both revenues and expenditures tend to rise and fall pro-cyclically.

The negative effects of the **interest rate** shock partly reflect higher interest payments on the debt, but also the fact that nominal expenditures are relatively stable even as inflation (and therefore revenues) decline.
